# Notebook 1: Tokenization, Embeddings, and Self-Attention
This notebook covers tokenization, embedding lookup, positional encoding, and a basic attention mechanism.

## 1. Subword Tokenization Example
Instead of just splitting on spaces or characters, modern NLP models (like LLaMA or BERT) use **subword tokenization**. Libraries like the [Hugging Face Tokenizers library](https://huggingface.co/docs/tokenizers/index) build these efficiently in Rust.

Below, we simulate mapping subword pieces (with WordPiece `##` prefixes) to their integer IDs.

In [ ]:
tokens = ['hello', 'trans', '##form', '##ers']
vocab = {'hello': 10, 'trans': 42, '##form': 89, '##ers': 3, '[UNK]': 0}

# TODO 1: Fill in the list comprehension to map tokens to their IDs using the vocab dictionary.
# If a token is not in vocab, return the ID for '[UNK]'.
# SOLUTION: token_ids = [vocab.get(tok, vocab['[UNK]']) for tok in tokens]
token_ids = [ ] # YOUR CODE HERE

print("Tokens:", tokens)
print("Token IDs:", token_ids)

## 2. Embedding Lookup vs One-Hot Equivalence
Given token IDs, embeddings are retrieved by table lookup. Mathematically, this is equivalent to multiplying a one-hot vector by the embedding matrix.

In [ ]:
import numpy as np
V, d = 5, 4
E = np.arange(V*d).reshape(V, d).astype(float)
print("Embedding matrix:\n", E)

token_id = 1
one_hot = np.zeros(V)
one_hot[token_id] = 1

# TODO 2: Compute the embedding using a dot product between 'one_hot' and 'E'.
# SOLUTION: emb_vector = np.dot(one_hot, E)
emb_vector = None # YOUR CODE HERE
print("One-hot @ E =\n", emb_vector)

## 3. Positional Encoding (Sinusoidal)
Since Transformers have no recurrence, we add sinusoidal position encodings to the tokens to inject order information.

In [ ]:
def sinusoidal_pos_enc(max_pos, d_model):
    pos = np.arange(max_pos)[:, None]
    i = np.arange(d_model)[None, :]
    angle_rates = 1 / np.power(10000, (2*(i//2))/np.float32(d_model))
    angle_rads = pos * angle_rates
    enc = np.zeros_like(angle_rads)
    enc[:, 0::2] = np.sin(angle_rads[:, 0::2])
    enc[:, 1::2] = np.cos(angle_rads[:, 1::2])
    return enc

enc = sinusoidal_pos_enc(5, 4)
print("Sinusoidal encodings (pos 0-4, d=4):\n", enc)

## 4. Scaled Dot-Product Attention
We compute the attention scores by taking the dot product of $Q$ and $K^T$, scaled by $\sqrt{d_k}$. For efficiency, modern models use [PyTorch MultiheadAttention](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) which utilizes optimized `torch.bmm` operations under the hood.

In [ ]:
Q = np.array([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
K = np.array([[1.0, 2.0, 3.0], [1.0, 0.0, 1.0]])
V = np.array([[0.1, 0.2], [0.3, 0.4]])

d_k = Q.shape[1]
scores = Q.dot(K.T) / np.sqrt(d_k)

def softmax(x):
    # TODO 3: Implement the softmax function (exponentiate x, then divide by the sum along axis 1).
    # Make sure to use keepdims=True for broadcasting.
    # SOLUTION: return np.exp(x) / np.sum(np.exp(x), axis=1, keepdims=True)
    return x # YOUR CODE HERE

probs = softmax(scores)
output = probs.dot(V) if probs is not scores else np.zeros_like(V)
print("Attention output:\n", output)

## 5. Attention Matrix Visualization
Visualizing the attention probability matrix as a heatmap shows exactly which keys a given query is "looking at".

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# We use a dummy probability matrix to guarantee a successful plot even if the TODO is unfinished.
plot_probs = np.array([[0.5, 0.5], [0.76, 0.24]]) 

plt.figure(figsize=(6, 4))
sns.heatmap(plot_probs, annot=True, cmap='viridis', xticklabels=['Key 1', 'Key 2'], yticklabels=['Query 1', 'Query 2'])
plt.xlabel('Keys')
plt.ylabel('Queries')
plt.title('Self-Attention Probability Heatmap')
plt.show()